# Build Daily Sales
Aggregates order data from the Silver Layer to create daily sales metrics for reporting and analysis.

In [0]:
CREATE OR REPLACE TABLE ecommerce_catalog.gold.daily_sales AS
WITH bounds AS (
    SELECT 
        MIN(DATE(order_date)) AS min_date,
        MAX(DATE(order_date)) AS max_date
    FROM ecommerce_catalog.silver.silver_orders
),
calendar AS (
    SELECT 
        EXPLODE(SEQUENCE(min_date, max_date, INTERVAL 1 DAY)) AS calendar_date
    FROM bounds
)
SELECT 
    c.calendar_date                                         AS sales_date,

    -- 2. Einfache Zählungen
    COUNT(o.order_id)                                       AS order_count,
    -- 3. Summen & Beträge (mit Null-Absicherung)
    COALESCE(SUM(o.quantity), 0)                            AS units_sold,
    
    ROUND(COALESCE(SUM(o.quantity * o.unit_price), 0.0),2)  AS revenue,
    COUNT(DISTINCT o.customer_id)                           AS unique_customers,    

    -- 4. Durchschnitte
    ROUND(COALESCE(AVG(o.quantity * o.unit_price), 0.0),2)  AS avg_order_value

FROM calendar c
LEFT JOIN ecommerce_catalog.silver.silver_orders o 
    ON c.calendar_date = DATE(o.order_date)
GROUP BY 
    c.calendar_date
ORDER BY 
    c.calendar_date ASC;